In [1]:
"""
Signaling Game with Occasionally Observed World States
(Numba-optimized version)

Changes in this version
-----------------------
* n=4, 6, 8 sweeps all run at 10^6 plays/run, 1000 runs per cell.
* For EVERY point on the graphs (variant x p value) three n x n matrices are
  produced, averaged over all runs:
    1. Sender   : rows = state seen,       cols = signal sent    P(signal | state)
    2. State    : rows = state observed,   cols = action         P(action | state)
    3. Receiver : rows = signal received,  cols = action         P(action | signal)
  (this is the n x n generalisation of the 2x2 matrices on the whiteboard:
   the ideal outcome is a permutation matrix of 1s/0s, the learned averages
   are the "0.8 / 0.2" style numbers.)
* Because each run picks its own arbitrary signal convention, a plain average
  of sender/receiver matrices over runs blurs toward uniform. So for those two
  matrices BOTH versions are saved:
    - "aligned": in each run, signals are relabelled so state k's dominant
                 signal becomes signal k (receiver rows permuted to match),
                 then averaged. This is the one that looks like the whiteboard.
    - "raw":     plain average, no relabelling.
  The state matrix needs no alignment (action k already means "state k").
* All the original graphs are still produced.
"""

import numpy as np
import numba
from numba import njit, prange
import matplotlib.pyplot as plt
from dataclasses import dataclass
from enum import IntEnum
import time
from datetime import datetime
import os

# ---- Parallelism / reproducibility ----
NUM_THREADS = None
BASE_SEED = -1        # -1 = random each time; any int >= 0 = reproducible results

# ---- Game parameters (used by RUN_SINGLE only) ----
N = 6
P_OBSERVE = 0
TIMESTEPS = 1_000_000
N_RUNS = 1000

# ---- Variant selection (used by RUN_SINGLE only) ----
ACTION_SELECTION = "SO"
REINFORCEMENT = "no_gating"

# ---- What to run ----
RUN_SINGLE = False
RUN_SCALING_TABLE = False
RUN_N4_SCALING = True     # n=4, 1000 runs, 10^6 plays per run
RUN_N6_SCALING = True     # n=6, 1000 runs, 10^6 plays per run
RUN_N8_SCALING = True     # n=8, 1000 runs, 10^6 plays per run
SCALING_T = 1_000_000
SCALING_RUNS = 1000


class ActionSelectionEnum(IntEnum):
    LP = 0
    BP = 1
    SO = 2


class ReinforcementEnum(IntEnum):
    NO_GATING = 0
    GATED = 1


def parse_action_selection(s: str) -> ActionSelectionEnum:
    mapping = {"LP": ActionSelectionEnum.LP, "BP": ActionSelectionEnum.BP, "SO": ActionSelectionEnum.SO}
    return mapping[s.upper()]


def parse_reinforcement(s: str) -> ReinforcementEnum:
    mapping = {"NO_GATING": ReinforcementEnum.NO_GATING, "GATED": ReinforcementEnum.GATED}
    return mapping[s.upper().replace("-", "_")]


@njit
def compute_normalized_entropy(probs):
    """Calculates entropy normalized by log(N) so it ranges from 0 to 1."""
    n = len(probs)
    if n <= 1: return 0.0
    ent = 0.0
    for p in probs:
        if p > 1e-12:
            ent -= p * np.log(p)
    return ent / np.log(n)


@njit
def compute_mixture_metrics(Q_sender, Q_recv_state):
    """
    1) State Mixture:       Avg entropy of P(signal|state)     [sender]
    2) Signal Mixture:      Avg entropy of P(state|signal)     [sender inverted]
    3) Recv State Mixture:  Avg entropy of P(action|state)     [receiver, state-only]
    """
    n = Q_sender.shape[0]

    sender_strat = np.zeros((n, n))
    for i in range(n):
        row_sum = Q_sender[i].sum()
        sender_strat[i] = Q_sender[i] / row_sum

    sum_state_ent = 0.0
    for i in range(n):
        sum_state_ent += compute_normalized_entropy(sender_strat[i])
    avg_state_mixture = sum_state_ent / n

    p_state_given_sig = np.zeros((n, n))   # [signal, state]
    for sig in range(n):
        col_sum = 0.0
        for st in range(n):
            col_sum += sender_strat[st, sig]
        if col_sum > 1e-12:
            for st in range(n):
                p_state_given_sig[sig, st] = sender_strat[st, sig] / col_sum
        else:
            for st in range(n):
                p_state_given_sig[sig, st] = 1.0 / n

    sum_sig_ent = 0.0
    for j in range(n):
        sum_sig_ent += compute_normalized_entropy(p_state_given_sig[j])
    avg_signal_mixture = sum_sig_ent / n

    sum_recv_state_ent = 0.0
    for i in range(n):
        row_sum = Q_recv_state[i].sum()
        recv_probs = Q_recv_state[i] / row_sum
        sum_recv_state_ent += compute_normalized_entropy(recv_probs)
    avg_recv_state_mixture = sum_recv_state_ent / n

    return avg_state_mixture, avg_signal_mixture, avg_recv_state_mixture


@njit
def choose_from_probs(probs):
    r = np.random.random()
    cumsum = 0.0
    for i in range(len(probs)):
        cumsum += probs[i]
        if r < cumsum: return i
    return len(probs) - 1


@njit
def run_single_simulation(n, p_observe, action_selection, reinforcement, timesteps):
    Q_sender = np.ones((n, n))
    Q_recv_signal = np.ones((n, n))
    Q_recv_state = np.ones((n, n))
    total_payoff = 0
    for t in range(timesteps):
        state = np.random.randint(0, n)
        sender_probs = Q_sender[state] / Q_sender[state].sum()
        signal = choose_from_probs(sender_probs)
        observe_state = np.random.random() < p_observe
        if observe_state:
            probs_state = Q_recv_state[state] / Q_recv_state[state].sum()
            probs_signal = Q_recv_signal[signal] / Q_recv_signal[signal].sum()
            rec_state = choose_from_probs(probs_state)
            rec_signal = choose_from_probs(probs_signal)
            if action_selection == 2: action = choose_from_probs(probs_state)
            elif action_selection == 0:
                pooled_probs = 0.5 * probs_state + 0.5 * probs_signal
                action = choose_from_probs(pooled_probs)
            else:
                pooled_Q = Q_recv_state[state] + Q_recv_signal[signal]
                action = choose_from_probs(pooled_Q / pooled_Q.sum())
            success = (action == state)
            if success:
                total_payoff += 1
                Q_sender[state, signal] += 1
                if reinforcement == 0:
                    Q_recv_state[state, action] += 1
                    Q_recv_signal[signal, action] += 1
                else:
                    if rec_state == action: Q_recv_state[state, action] += 1
                    if rec_signal == action: Q_recv_signal[signal, action] += 1
        else:
            probs_signal = Q_recv_signal[signal] / Q_recv_signal[signal].sum()
            action = choose_from_probs(probs_signal)
            success = (action == state)
            if success:
                total_payoff += 1
                Q_sender[state, signal] += 1
                Q_recv_signal[signal, action] += 1
    return Q_sender, Q_recv_signal, Q_recv_state, total_payoff


@njit
def compute_expected_payoff(n, p_observe, action_selection, Q_sender, Q_recv_signal, Q_recv_state):
    sender_strat = np.zeros((n, n))
    for i in range(n): sender_strat[i] = Q_sender[i] / Q_sender[i].sum()
    recv_signal_strat = np.zeros((n, n))
    recv_state_strat = np.zeros((n, n))
    for i in range(n):
        recv_signal_strat[i] = Q_recv_signal[i] / Q_recv_signal[i].sum()
        recv_state_strat[i] = Q_recv_state[i] / Q_recv_state[i].sum()
    exp_signal_only = 0.0
    for i in range(n):
        for j in range(n): exp_signal_only += sender_strat[i, j] * recv_signal_strat[j, i]
    exp_signal_only /= n
    exp_state_obs = 0.0
    for i in range(n):
        for j in range(n):
            p_sig = sender_strat[i, j]
            if action_selection == 2: p_correct = recv_state_strat[i, i]
            elif action_selection == 0: p_correct = 0.5 * recv_state_strat[i, i] + 0.5 * recv_signal_strat[j, i]
            else:
                Q_pooled = Q_recv_state[i] + Q_recv_signal[j]
                p_correct = Q_pooled[i] / Q_pooled.sum()
            exp_state_obs += p_sig * p_correct
    exp_state_obs /= n
    return (1 - p_observe) * exp_signal_only + p_observe * exp_state_obs


@njit
def align_signals(sender_strat):
    """
    Greedy one-to-one matching of states to signals by sender probability.
    Returns perm with perm[k] = signal assigned to state k.
    """
    n = sender_strat.shape[0]
    perm = np.full(n, -1, dtype=np.int64)
    used_state = np.zeros(n, dtype=np.bool_)
    used_sig = np.zeros(n, dtype=np.bool_)
    for _ in range(n):
        best = -1.0
        bi = -1
        bj = -1
        for i in range(n):
            if used_state[i]:
                continue
            for j in range(n):
                if used_sig[j]:
                    continue
                if sender_strat[i, j] > best:
                    best = sender_strat[i, j]
                    bi = i
                    bj = j
        perm[bi] = bj
        used_state[bi] = True
        used_sig[bj] = True
    return perm


@njit(parallel=True)
def run_multiple_simulations_mats(n, p_observe, action_selection, reinforcement, timesteps,
                                  n_runs, seed=-1):
    """
    Same as run_multiple_simulations but ALSO returns, per run, the final
    strategy matrices (row-normalised to probabilities):
        sender_raw   [run, state, signal]
        recv_raw     [run, signal, action]
        state_mat    [run, state, action]
        sender_al    [run, state, signal-aligned-to-state]
        recv_al      [run, signal-aligned-to-state, action]
    """
    final_expected_payoffs = np.zeros(n_runs)
    average_payoffs        = np.zeros(n_runs)
    state_mixtures         = np.zeros(n_runs)
    signal_mixtures        = np.zeros(n_runs)
    recv_state_mixtures    = np.zeros(n_runs)
    sender_raw = np.zeros((n_runs, n, n))
    recv_raw   = np.zeros((n_runs, n, n))
    state_mat  = np.zeros((n_runs, n, n))
    sender_al  = np.zeros((n_runs, n, n))
    recv_al    = np.zeros((n_runs, n, n))

    for run in prange(n_runs):
        if seed >= 0:
            np.random.seed(seed + run)
        Q_sender, Q_recv_signal, Q_recv_state, total_payoff = run_single_simulation(
            n, p_observe, action_selection, reinforcement, timesteps
        )
        final_expected_payoffs[run] = compute_expected_payoff(
            n, p_observe, action_selection, Q_sender, Q_recv_signal, Q_recv_state
        )
        average_payoffs[run] = total_payoff / timesteps
        st_mix, sig_mix, rs_mix = compute_mixture_metrics(Q_sender, Q_recv_state)
        state_mixtures[run]      = st_mix
        signal_mixtures[run]     = sig_mix
        recv_state_mixtures[run] = rs_mix

        S = np.zeros((n, n))
        R = np.zeros((n, n))
        T = np.zeros((n, n))
        for i in range(n):
            S[i] = Q_sender[i] / Q_sender[i].sum()
            R[i] = Q_recv_signal[i] / Q_recv_signal[i].sum()
            T[i] = Q_recv_state[i] / Q_recv_state[i].sum()
        perm = align_signals(S)
        for k in range(n):
            for j in range(n):
                sender_raw[run, k, j] = S[k, j]
                recv_raw[run, k, j]   = R[k, j]
                state_mat[run, k, j]  = T[k, j]
                sender_al[run, k, j]  = S[k, perm[j]]   # column j = signal assigned to state j
                recv_al[run, k, j]    = R[perm[k], j]   # row k    = signal assigned to state k

    return (final_expected_payoffs, average_payoffs, state_mixtures, signal_mixtures,
            recv_state_mixtures, sender_raw, recv_raw, state_mat, sender_al, recv_al)


@njit(parallel=True)
def run_multiple_simulations(n, p_observe, action_selection, reinforcement, timesteps, n_runs,
                             seed=-1):
    final_expected_payoffs = np.zeros(n_runs)
    average_payoffs        = np.zeros(n_runs)
    state_mixtures         = np.zeros(n_runs)
    signal_mixtures        = np.zeros(n_runs)
    recv_state_mixtures    = np.zeros(n_runs)

    for run in prange(n_runs):
        if seed >= 0:
            np.random.seed(seed + run)
        Q_sender, Q_recv_signal, Q_recv_state, total_payoff = run_single_simulation(
            n, p_observe, action_selection, reinforcement, timesteps
        )
        final_expected_payoffs[run] = compute_expected_payoff(
            n, p_observe, action_selection, Q_sender, Q_recv_signal, Q_recv_state
        )
        average_payoffs[run] = total_payoff / timesteps
        st_mix, sig_mix, rs_mix = compute_mixture_metrics(Q_sender, Q_recv_state)
        state_mixtures[run]      = st_mix
        signal_mixtures[run]     = sig_mix
        recv_state_mixtures[run] = rs_mix

    return final_expected_payoffs, average_payoffs, state_mixtures, signal_mixtures, recv_state_mixtures


@dataclass
class GameConfig:
    n: int = 2
    p_observe: float = 0.1
    action_selection: ActionSelectionEnum = ActionSelectionEnum.LP
    reinforcement: ReinforcementEnum = ReinforcementEnum.NO_GATING
    timesteps: int = 10000
    n_runs: int = 100

    def variant_name(self) -> str:
        a_names = {0: "LP", 1: "BP", 2: "SO"}
        r_names = {0: "no_gating", 1: "gated"}
        return f"{a_names[int(self.action_selection)]}, {r_names[int(self.reinforcement)]}"

    @classmethod
    def from_globals(cls):
        return cls(n=N, p_observe=P_OBSERVE, timesteps=TIMESTEPS, n_runs=N_RUNS,
                   action_selection=parse_action_selection(ACTION_SELECTION),
                   reinforcement=parse_reinforcement(REINFORCEMENT))


def run_experiments(config: GameConfig, verbose: bool = True) -> dict:
    if verbose: print(f"Running {config.n_runs} simulations...", flush=True)
    start = time.time()

    final_payoffs, avg_payoffs, st_mixes, sig_mixes, rs_mixes = run_multiple_simulations(
        config.n, config.p_observe, int(config.action_selection),
        int(config.reinforcement), config.timesteps, config.n_runs, BASE_SEED
    )

    if verbose: print(f"Completed in {time.time() - start:.2f}s", flush=True)

    return {
        'final_expected_payoffs':   final_payoffs,
        'average_payoffs':          avg_payoffs,
        'state_mixtures':           st_mixes,
        'signal_mixtures':          sig_mixes,
        'recv_state_mixtures':      rs_mixes,
        'success_rate':             np.mean(final_payoffs >= 0.875),
        'mean_final_expected':      np.mean(final_payoffs),
        'mean_average_payoff':      np.mean(avg_payoffs),
        'mean_state_mixture':       np.mean(st_mixes),
        'mean_signal_mixture':      np.mean(sig_mixes),
        'mean_recv_state_mixture':  np.mean(rs_mixes),
    }


def _fmt_timestep(t: int) -> str:
    exponent = round(np.log10(t))
    if 10 ** exponent == t: return f"1e{exponent}"
    return f"{t:,}"


def _save_cell_distribution_plot(final_payoffs, variant_name, n, timesteps, p_obs, n_runs,
                                  success_rate, mean_avg_payoff, st_mix, sig_mix, rs_mix, save_path):
    plt.style.use("default")
    fig, ax = plt.subplots(figsize=(8, 5), facecolor="white")
    ax.hist(final_payoffs, bins=20, density=True, alpha=0.7, color="steelblue", edgecolor="white")
    ax.axvline(x=0.875, color="red", linestyle="--", label=f"Threshold 0.875")
    ax.set_title(
        f"{variant_name} | n={n}, T={timesteps:,}, p={p_obs:.1f}\n"
        f"Success: {success_rate*100:.1f}% | Avg Payoff: {mean_avg_payoff:.3f}\n"
        f"State Mix: {st_mix:.3f} | Signal Mix: {sig_mix:.3f} | Recv-State Mix: {rs_mix:.3f}",
        fontsize=9
    )
    ax.legend(fontsize=8)
    ax.set_xlim(0, 1.05)
    plt.tight_layout()
    plt.savefig(save_path, dpi=120)
    plt.close()


def _n_runs_for_timestep(t: int) -> int:
    return 100 if t >= 10_000_000 else 1_000


def run_scaling_table(n=4, save_dir="scaling_results"):
    t_grid = [100, 1_000, 10_000, 100_000, 1_000_000, 10_000_000]
    p_grid = [0.0, 0.1, 0.3, 0.5, 0.7, 0.9, 1.0]
    variants = [
        ("LP", "no_gating", 0, 0), ("LP", "gated", 0, 1),
        ("BP", "no_gating", 1, 0), ("BP", "gated", 1, 1),
        ("SO", "no_gating", 2, 0), ("SO", "gated", 2, 1)
    ]

    os.makedirs(save_dir, exist_ok=True)
    summary_path = os.path.join(save_dir, "all_metrics_summary.txt")

    COL_W = 22
    PLAYS_W = 8

    def col_header(p):
        label = f"p={p:.1f}"
        return label.center(COL_W - 1) + " "

    def col_header_sub():
        sub = "Suc% Pay StMx SgMx RsMx"
        return sub.center(COL_W - 1) + " "

    def divider(n_p):
        return "-" * (PLAYS_W + 3 + n_p * (COL_W + 1))

    with open(summary_path, "w") as f_sum:
        f_sum.write(f"SCALING TABLE  n={n}  {datetime.now()}\n")
        f_sum.write("Metrics: Suc%=success rate  Pay=avg payoff  "
                    "StMx=state mixture  SgMx=signal mixture  RsMx=recv-state mixture\n"
                    "RsMx: 0=receiver knows exactly what to do, 1=no idea\n\n")

        for action_label, reinf_label, a_sel, reinf in variants:
            v_name = f"{action_label}, {reinf_label}"
            f_sum.write(f"{'='*80}\nVARIANT: {v_name}\n{'='*80}\n")

            header1 = f"{'Plays':<{PLAYS_W}} | " + "| ".join(col_header(p) for p in p_grid)
            header2 = f"{'':>{PLAYS_W}} | " + "| ".join(col_header_sub() for _ in p_grid)
            div     = divider(len(p_grid))
            f_sum.write(header1 + "\n")
            f_sum.write(header2 + "\n")
            f_sum.write(div     + "\n")

            for t in t_grid:
                current_n_runs = _n_runs_for_timestep(t)
                row_str = f"{_fmt_timestep(t):<{PLAYS_W}} | "

                for p in p_grid:
                    res, avgs, st, sig, rs = run_multiple_simulations(
                        n, p, a_sel, reinf, t, current_n_runs, BASE_SEED
                    )

                    s_rate = np.mean(res >= 0.875) * 100
                    a_pay  = np.mean(avgs)
                    m_st   = np.mean(st)
                    m_sig  = np.mean(sig)
                    m_rs   = np.mean(rs)

                    cell = f"{s_rate:4.0f}% {a_pay:.2f} {m_st:.2f} {m_sig:.2f} {m_rs:.2f} "
                    row_str += cell + "| "

                    dist_dir = os.path.join(save_dir, "plots", v_name.replace(", ", "_"))
                    os.makedirs(dist_dir, exist_ok=True)
                    _save_cell_distribution_plot(
                        res, v_name, n, t, p, current_n_runs,
                        s_rate / 100, a_pay, m_st, m_sig, m_rs,
                        os.path.join(dist_dir, f"T{_fmt_timestep(t)}_p{p:.1f}.png")
                    )

                f_sum.write(row_str + "\n")

            f_sum.write("\n")

    print(f"Scaling table saved to: {summary_path}")


# ─────────────────────────────────────────────────────────────────────────────
# Matrix output (sender / state / receiver) for every (variant, T, p)
# ─────────────────────────────────────────────────────────────────────────────
def _matrix_to_text(M, row_label, col_label):
    n = M.shape[0]
    head = f"{row_label} \\ {col_label}"
    lines = [f"    {head}", "        " + " ".join(f"{j:>5d}" for j in range(n))]
    for i in range(n):
        lines.append(f"    {i:>3d} " + " ".join(f"{M[i, j]:5.2f}" for j in range(n)))
    return "\n".join(lines)


def _draw_matrix(ax, M, title, row_label, col_label):
    n = M.shape[0]
    ax.imshow(M, vmin=0, vmax=1, cmap="Blues")
    ax.set_title(title, fontsize=9)
    ax.set_xlabel(col_label, fontsize=8)
    ax.set_ylabel(row_label, fontsize=8)
    ax.set_xticks(range(n))
    ax.set_yticks(range(n))
    ax.tick_params(labelsize=7)
    for i in range(n):
        for j in range(n):
            ax.text(j, i, f"{M[i, j]:.2f}", ha="center", va="center", fontsize=7,
                    color="white" if M[i, j] > 0.55 else "black")


def _save_matrix_figure(mats, v_name, n, t, p, n_runs, path):
    """Top row: aligned sender / state / aligned receiver. Bottom row: raw sender / raw receiver."""
    fig, axes = plt.subplots(2, 3, figsize=(13, 8.5), facecolor="white")
    _draw_matrix(axes[0, 0], mats["sender_aligned"], "1. Sender (aligned)\nP(sent | seen)",
                 "state seen", "signal sent (relabelled)")
    _draw_matrix(axes[0, 1], mats["state"], "2. State\nP(act | obs)",
                 "state observed", "action")
    _draw_matrix(axes[0, 2], mats["receiver_aligned"], "3. Receiver (aligned)\nP(act | rec)",
                 "signal received (relabelled)", "action")
    _draw_matrix(axes[1, 0], mats["sender_raw"], "Sender (raw average)",
                 "state seen", "signal sent")
    axes[1, 1].axis("off")
    _draw_matrix(axes[1, 2], mats["receiver_raw"], "Receiver (raw average)",
                 "signal received", "action")
    fig.suptitle(f"{v_name} | n={n}, T={_fmt_timestep(t)}, p={p:.1f} | mean of {n_runs} runs",
                 fontsize=11)
    plt.tight_layout(rect=(0, 0, 1, 0.96))
    plt.savefig(path, dpi=110)
    plt.close()


def _make_sweep_plot(all_results, variants, p_grid, t, n, metric_keys, ylabel,
                     title_suffix, filepath, label_map=None):
    """One line per variant (LP/BP/SO x no_gating/gated). If two metric keys are
    given, the second is drawn fainter."""
    COLOR_MAP  = {"LP": "tab:blue", "BP": "tab:orange", "SO": "tab:green"}
    DASH_MAP   = {"no_gating": "-", "gated": "--"}
    MARKER_MAP = {"no_gating": "o", "gated": "s"}
    if isinstance(metric_keys, str):
        metric_keys = [metric_keys]

    fig, ax = plt.subplots(figsize=(9, 5), facecolor="white")
    for action_label, reinf_label, _, _ in variants:
        v_name = f"{action_label}, {reinf_label}"
        for idx, mkey in enumerate(metric_keys):
            vals = [all_results[v_name][t][p][mkey] for p in p_grid]
            name = f"{v_name}" if idx == 0 else f"  ↳ {label_map[mkey]} ({v_name})"
            ax.plot(p_grid, vals,
                    color=COLOR_MAP[action_label], linestyle=DASH_MAP[reinf_label],
                    marker=MARKER_MAP[reinf_label], markersize=5,
                    linewidth=1.8 if idx == 0 else 1.2,
                    alpha=1.0 if idx == 0 else 0.45, label=name)
    ax.set_xlabel("p_observe", fontsize=11)
    ax.set_ylabel(ylabel, fontsize=11)
    ax.set_title(f"n={n}, T={_fmt_timestep(t)} — {title_suffix}\n"
                 "Color: LP=blue  BP=orange  SO=green   Line: solid=no_gating  dashed=gated",
                 fontsize=9)
    ax.legend(fontsize=7, ncol=2, loc="best")
    ax.set_xlim(-0.02, 1.02)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(filepath, dpi=130)
    plt.close()


def run_scaling_sweep(n: int, t_grid: list, n_runs: int = 1000,
                      save_dir: str = "scaling_sweep",
                      p_grid=(0.0, 0.1, 0.3, 0.5, 0.7, 0.9, 1.0),
                      make_plots: bool = True,
                      save_matrices: bool = True) -> dict:
    """
    Runs all 6 variants (LP/BP/SO x no_gating/gated) for every (T, p) combination,
    writes a summary table, all sweep graphs, and (per variant/T/p) the three
    mean strategy matrices: sender, state, receiver.
    Returns all_results[variant][T][p] -> dict of metrics (+ 'matrices').
    """
    p_grid = list(p_grid)
    variants = [
        ("LP", "no_gating", 0, 0), ("LP", "gated", 0, 1),
        ("BP", "no_gating", 1, 0), ("BP", "gated", 1, 1),
        ("SO", "no_gating", 2, 0), ("SO", "gated", 2, 1),
    ]
    os.makedirs(save_dir, exist_ok=True)
    summary_path = os.path.join(save_dir, f"n{n}_summary.txt")
    matrices_txt_path = os.path.join(save_dir, f"n{n}_matrices.txt")
    matrices_npz_path = os.path.join(save_dir, f"n{n}_matrices.npz")
    matrix_plot_dir = os.path.join(save_dir, "matrices")
    if save_matrices:
        os.makedirs(matrix_plot_dir, exist_ok=True)

    PLAYS_W = 8
    CELL_W  = 25

    def col_header(p):  return f"p={p:.1f}".center(CELL_W)
    def col_sub():      return "Suc% Pay StMx SgMx RsMx".center(CELL_W)
    total_w = PLAYS_W + 3 + len(p_grid) * (CELL_W + 3)
    div = "-" * total_w
    bar = "=" * total_w

    all_results = {}
    npz_store = {}

    f_mat = open(matrices_txt_path, "w") if save_matrices else None
    if f_mat:
        f_mat.write(f"MEAN STRATEGY MATRICES  n={n}  {datetime.now()}\n"
                    f"Each matrix is averaged over {n_runs} runs. Rows sum to 1.\n"
                    "Sender   : rows = state seen,     cols = signal sent\n"
                    "State    : rows = state observed, cols = action\n"
                    "Receiver : rows = signal received, cols = action\n"
                    "'aligned' = signals relabelled per run so state k's dominant signal is k.\n\n")

    with open(summary_path, "w") as f:
        f.write(f"SCALING TABLE  n={n}  {datetime.now()}\n")
        f.write("Metrics: Suc%=success rate  Pay=avg payoff  "
                "StMx=state mixture  SgMx=signal mixture  RsMx=recv-state mixture\n")
        f.write("RsMx: 0=receiver knows exactly what to do, 1=no idea\n")
        f.write(f"Runs per cell: {n_runs}\n\n")

        for action_label, reinf_label, a_sel, reinf in variants:
            v_name = f"{action_label}, {reinf_label}"
            v_tag = v_name.replace(", ", "_")
            all_results[v_name] = {}

            header1 = f"{'Plays':<{PLAYS_W}} | " + " | ".join(col_header(p) for p in p_grid) + " |"
            header2 = f"{'':>{PLAYS_W}} | " + " | ".join(col_sub() for _ in p_grid) + " |"

            block = [bar, f"VARIANT: {v_name}", bar, header1, header2, div]
            print(f"\nVARIANT: {v_name}", flush=True)

            for t in t_grid:
                all_results[v_name][t] = {}
                row = f"{_fmt_timestep(t):<{PLAYS_W}} | "

                for p in p_grid:
                    (res, avgs, st, sig, rs,
                     S_raw, R_raw, St, S_al, R_al) = run_multiple_simulations_mats(
                        n, p, a_sel, reinf, t, n_runs, BASE_SEED
                    )
                    mats = {
                        "sender_aligned":   S_al.mean(axis=0),
                        "state":            St.mean(axis=0),
                        "receiver_aligned": R_al.mean(axis=0),
                        "sender_raw":       S_raw.mean(axis=0),
                        "receiver_raw":     R_raw.mean(axis=0),
                    }
                    m = {
                        "success_rate":            float(np.mean(res >= 0.875)),
                        "mean_avg_payoff":         float(np.mean(avgs)),
                        "mean_exp_payoff":         float(np.mean(res)),
                        "mean_state_mixture":      float(np.mean(st)),
                        "mean_signal_mixture":     float(np.mean(sig)),
                        "mean_recv_state_mixture": float(np.mean(rs)),
                        "final_payoffs":           res,
                        "matrices":                mats,
                    }
                    all_results[v_name][t][p] = m

                    cell = (f"{m['success_rate']*100:4.0f}% {m['mean_avg_payoff']:.2f} "
                            f"{m['mean_state_mixture']:.2f} {m['mean_signal_mixture']:.2f} "
                            f"{m['mean_recv_state_mixture']:.2f}")
                    row += f"{cell:<{CELL_W}} | "

                    if save_matrices:
                        key = f"{v_tag}__T{_fmt_timestep(t)}__p{p:.1f}"
                        for name, M in mats.items():
                            npz_store[f"{key}__{name}"] = M
                        f_mat.write(f"{'='*60}\nVARIANT: {v_name}   T={_fmt_timestep(t)}   p={p:.1f}\n{'='*60}\n")
                        f_mat.write("1. SENDER (aligned)  P(sent | seen)\n" +
                                    _matrix_to_text(mats["sender_aligned"], "seen", "sent") + "\n")
                        f_mat.write("2. STATE  P(act | obs)\n" +
                                    _matrix_to_text(mats["state"], "obs", "act") + "\n")
                        f_mat.write("3. RECEIVER (aligned)  P(act | rec)\n" +
                                    _matrix_to_text(mats["receiver_aligned"], "rec", "act") + "\n")
                        f_mat.write("   (raw, unaligned) SENDER\n" +
                                    _matrix_to_text(mats["sender_raw"], "seen", "sent") + "\n")
                        f_mat.write("   (raw, unaligned) RECEIVER\n" +
                                    _matrix_to_text(mats["receiver_raw"], "rec", "act") + "\n\n")
                        f_mat.flush()

                        vdir = os.path.join(matrix_plot_dir, v_tag)
                        os.makedirs(vdir, exist_ok=True)
                        _save_matrix_figure(mats, v_name, n, t, p, n_runs,
                                            os.path.join(vdir, f"T{_fmt_timestep(t)}_p{p:.1f}.png"))

                block.append(row)
                print(f"  T={_fmt_timestep(t)} done", flush=True)

            block.append("")
            text = "\n".join(block) + "\n"
            f.write(text)
            f.flush()
            print(text)

    if f_mat:
        f_mat.close()
        np.savez_compressed(matrices_npz_path, **npz_store)
        print(f"Matrices saved to: {matrices_txt_path}, {matrices_npz_path}, {matrix_plot_dir}/")

    print(f"Summary saved to: {summary_path}")

    if make_plots:
        plots_dir = os.path.join(save_dir, "plots")
        os.makedirs(plots_dir, exist_ok=True)
        for t in t_grid:
            tl = _fmt_timestep(t)
            _make_sweep_plot(all_results, variants, p_grid, t, n, "success_rate",
                             "Success Rate", "Success Rate vs p_observe",
                             os.path.join(plots_dir, f"T{tl}_success_rate.png"))
            _make_sweep_plot(all_results, variants, p_grid, t, n, "mean_avg_payoff",
                             "Avg Payoff (over run)", "Average Payoff vs p_observe",
                             os.path.join(plots_dir, f"T{tl}_avg_payoff.png"))
            _make_sweep_plot(all_results, variants, p_grid, t, n, "mean_exp_payoff",
                             "Expected Payoff (final strategy)", "Expected Payoff vs p_observe",
                             os.path.join(plots_dir, f"T{tl}_exp_payoff.png"))
            _make_sweep_plot(all_results, variants, p_grid, t, n,
                             ["mean_state_mixture", "mean_signal_mixture"],
                             "Normalized Entropy", "Sender Entropy vs p_observe",
                             os.path.join(plots_dir, f"T{tl}_sender_entropy.png"),
                             label_map={"mean_state_mixture": "StateMix",
                                        "mean_signal_mixture": "SigMix"})
            _make_sweep_plot(all_results, variants, p_grid, t, n, "mean_recv_state_mixture",
                             "Normalized Entropy", "Receiver State-Mixture Entropy vs p_observe",
                             os.path.join(plots_dir, f"T{tl}_receiver_entropy.png"))
        print(f"Plots saved to: {plots_dir}")

    return all_results


def run_n4_scaling(n_runs: int = SCALING_RUNS, save_dir: str = "n4_scaling") -> dict:
    """n=4, 1000 runs, 10^6 plays per run."""
    return run_scaling_sweep(4, [SCALING_T], n_runs, save_dir)


def run_n6_scaling(n_runs: int = SCALING_RUNS, save_dir: str = "n6_scaling") -> dict:
    """n=6, 1000 runs, 10^6 plays per run."""
    return run_scaling_sweep(6, [SCALING_T], n_runs, save_dir)


def run_n8_scaling(n_runs: int = SCALING_RUNS, save_dir: str = "n8_scaling") -> dict:
    """n=8, 1000 runs, 10^6 plays per run."""
    return run_scaling_sweep(8, [SCALING_T], n_runs, save_dir)


def main():
    if NUM_THREADS is not None:
        numba.set_num_threads(NUM_THREADS)
    print(f"Numba threads in use: {numba.get_num_threads()} "
          f"(max {numba.config.NUMBA_NUM_THREADS}), threading layer requested: "
          f"{numba.config.THREADING_LAYER}")
    session_id = datetime.now().strftime("%Y%m%d_%H%M%S")
    base_dir = f"session_{session_id}"
    os.makedirs(base_dir, exist_ok=True)
    print(f"All results will be saved in: {base_dir}")

    # JIT warmup (both kernels)
    print("Warming up JIT...", end=" ", flush=True)
    run_multiple_simulations(2, 0.1, 0, 0, 100, 2, BASE_SEED)
    run_multiple_simulations_mats(2, 0.1, 0, 0, 100, 2, BASE_SEED)
    print("Done.\n")

    if RUN_SINGLE:
        config = GameConfig.from_globals()
        results = run_experiments(config)
        print(f"\nResults for {config.variant_name()}:")
        print(f"  Success Rate:        {results['success_rate']*100:.1f}%")
        print(f"  Avg Payoff:          {results['mean_average_payoff']:.4f}")
        print(f"  State Mixture:       {results['mean_state_mixture']:.4f}")
        print(f"  Signal Mixture:      {results['mean_signal_mixture']:.4f}")
        print(f"  Recv-State Mixture:  {results['mean_recv_state_mixture']:.4f}")

    if RUN_SCALING_TABLE:
        print("\nRunning Scaling Tables (this may take a while)...")
        run_scaling_table(n=N, save_dir=os.path.join(base_dir, "scaling"))

    if RUN_N4_SCALING:
        run_n4_scaling(save_dir=os.path.join(base_dir, "n4_scaling"))

    if RUN_N6_SCALING:
        run_n6_scaling(save_dir=os.path.join(base_dir, "n6_scaling"))

    if RUN_N8_SCALING:
        run_n8_scaling(save_dir=os.path.join(base_dir, "n8_scaling"))


if __name__ == "__main__":
    main()

Numba threads in use: 10 (max 10), threading layer requested: default
All results will be saved in: session_20261018_215840
Warming up JIT... Done.


VARIANT: LP, no_gating
  T=1e6 done
VARIANT: LP, no_gating
Plays    |           p=0.0           |           p=0.1           |           p=0.3           |           p=0.5           |           p=0.7           |           p=0.9           |           p=1.0           |
         |  Suc% Pay StMx SgMx RsMx  |  Suc% Pay StMx SgMx RsMx  |  Suc% Pay StMx SgMx RsMx  |  Suc% Pay StMx SgMx RsMx  |  Suc% Pay StMx SgMx RsMx  |  Suc% Pay StMx SgMx RsMx  |  Suc% Pay StMx SgMx RsMx  |
---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
1e6      |   79% 0.94 0.02 0.03 1.00 |   80% 0.95 0.02 0.03 0.00 |   80% 0.94 0.03 0.04 0.00 |   78% 0.94 0.05 0.06 0.00 |   73% 0.92 0.09 0.11 0.00 |   65% 0.90 0.16 0.